# Phase 4 (part 2) — Multi-Index & Cross-Stream Search

Indexes **docs + commits + issues** into a unified `evidence` index, then fuses them with the code index so one query returns citable evidence from all streams.

**Run first:** `uv run python -m archaeologist.indexing.streams_run`

This is the archaeologist's promise: an answer backed by a commit, a doc, an issue, AND code — not just one.

In [ ]:
import os, sys
from pathlib import Path

root = Path.cwd()
while root != root.parent and not (root / "pyproject.toml").exists():
    root = root.parent
os.chdir(root)
sys.path.insert(0, str(root / "src"))

from archaeologist.indexing import code_index, evidence_index
from archaeologist.indexing.opensearch_client import get_client
from archaeologist.retrieval.embeddings import get_embedder
from archaeologist.retrieval.multi import search_all
client = get_client()
embedder = get_embedder()
print("embedder:", type(embedder).__name__ if embedder else "NONE")

## What's indexed, by stream

In [ ]:
code_n = client.count(index=code_index.SYMBOL_INDEX)["count"]
print(f"  code (code_symbols): {code_n}")
for stream in ["doc", "commit", "issue"]:
    n = client.count(index=evidence_index.EVIDENCE_INDEX,
                     body={"query": {"term": {"stream": stream}}})["count"]
    print(f"  {stream:19}: {n}")

In [ ]:
def show(query, k=8, streams=None):
    tag = f" [{','.join(streams)}]" if streams else ""
    print(f"\n########## {query!r}{tag}\n")
    for hit in search_all(client, embedder, query, k=k, streams=streams):
        print(f"  {hit['score']:.4f}  [{hit['stream']:6}] {hit['citation']:26.26} {hit['title'][:46]}")

## Cross-stream: one question, evidence from everywhere

In [ ]:
show("why was async support added to views")
show("how is the application context and thread safety handled")
show("blueprint registration and nesting")

## Stream-filtered — ask only the git history, or only the issue tracker

In [ ]:
show("deprecation and removal of old APIs", k=5, streams=["commit"])
show("feature request or bug about typing", k=5, streams=["issue"])

## Summary

In [ ]:
ev = client.count(index=evidence_index.EVIDENCE_INDEX)["count"]
code_n = client.count(index=code_index.SYMBOL_INDEX)["count"]
print("Phase 4b —", "MULTI-INDEX OK ✅" if ev > 0 else "NO EVIDENCE ❌")
print(f"  evidence docs (doc+commit+issue): {ev}")
print(f"  code symbols                    : {code_n}")
print(f"  streams searchable              : code, doc, commit, issue")